# Model Training
Train a CNN classifier on modulation spectrograms. Target: >90% validation accuracy.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import custom modules
from src.data_preprocessing.load_data import load_dataset, partition_dataset
from src.data_preprocessing.create_spectrograms import batch_iq_to_spectrograms
from src.models.cnn_classifier import CNNClassifier
from src.models.train_utils import prepare_data_loaders, train_model, plot_training_history, save_training_config

print("Imports successful!")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load and Prepare Data

In [ ]:
# Load synthetic dataset
data_dir = Path('../data/synthetic')
dataset = load_dataset(str(data_dir))
print(f"Loaded {len(dataset)} samples")

# Partition into train/val/test
train_set, val_set, test_set = partition_dataset(dataset, train_ratio=0.7, val_ratio=0.15, seed=42)
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

# Create modulation-to-label mapping
modulations = sorted(set(s['modulation'] for s in dataset))
mod_to_label = {mod: idx for idx, mod in enumerate(modulations)}
label_to_mod = {idx: mod for mod, idx in mod_to_label.items()}
print(f"Modulations: {modulations}")
print(f"Number of classes: {len(modulations)}")

In [ ]:
# Convert IQ to spectrograms
print("Converting training spectrograms...")
train_specs = batch_iq_to_spectrograms(train_set, n_fft=256)
print(f"Train specs shape: {train_specs.shape}")

print("Converting validation spectrograms...")
val_specs = batch_iq_to_spectrograms(val_set, n_fft=256)
print(f"Val specs shape: {val_specs.shape}")

print("Converting test spectrograms...")
test_specs = batch_iq_to_spectrograms(test_set, n_fft=256)
print(f"Test specs shape: {test_specs.shape}")

# Create labels
train_labels = np.array([mod_to_label[s['modulation']] for s in train_set])
val_labels = np.array([mod_to_label[s['modulation']] for s in val_set])
test_labels = np.array([mod_to_label[s['modulation']] for s in test_set])

print(f"Label distributions:")
print(f"  Train: {np.bincount(train_labels)}")
print(f"  Val: {np.bincount(val_labels)}")

## Normalize Spectrograms

In [ ]:
# Normalize spectrograms to [0, 1]
def normalize_specs(specs):
    spec_min = specs.min()
    spec_max = specs.max()
    return (specs - spec_min) / (spec_max - spec_min + 1e-8)

train_specs_norm = normalize_specs(train_specs)
val_specs_norm = normalize_specs(val_specs)
test_specs_norm = normalize_specs(test_specs)

print("Normalized spectrogram stats:")
print(f"  Train: min={train_specs_norm.min():.4f}, max={train_specs_norm.max():.4f}, mean={train_specs_norm.mean():.4f}")
print(f"  Val: min={val_specs_norm.min():.4f}, max={val_specs_norm.max():.4f}, mean={val_specs_norm.mean():.4f}")

# Add channel dimension: (samples, height, width) -> (samples, 1, height, width)
train_specs_norm = train_specs_norm[:, np.newaxis, :, :]
val_specs_norm = val_specs_norm[:, np.newaxis, :, :]
test_specs_norm = test_specs_norm[:, np.newaxis, :, :]

print(f"Final shapes with channel:")
print(f"  Train: {train_specs_norm.shape}")
print(f"  Val: {val_specs_norm.shape}")
print(f"  Test: {test_specs_norm.shape}")

## Create DataLoaders

In [ ]:
batch_size = 32
train_loader, val_loader = prepare_data_loaders(
    train_specs_norm, train_labels,
    val_specs_norm, val_labels,
    batch_size=batch_size
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Check one batch
for specs_batch, labels_batch in train_loader:
    print(f"\nBatch specs shape: {specs_batch.shape}")
    print(f"Batch labels shape: {labels_batch.shape}")
    print(f"Sample labels: {labels_batch[:5]}")
    break

## Initialize Model

In [ ]:
num_classes = len(modulations)
model = CNNClassifier(in_channels=1, num_classes=num_classes, dropout_rate=0.5)
model = model.to(device)

print(f"Model created with {num_classes} output classes")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Print model architecture
print("\nModel architecture:")
print(model)

## Train Model

In [ ]:
# Training configuration
config = {
    'num_epochs': 50,
    'batch_size': batch_size,
    'learning_rate': 1e-3,
    'num_classes': num_classes,
    'modulation_classes': modulations,
    'train_samples': len(train_set),
    'val_samples': len(val_set),
    'device': str(device)
}

checkpoint_dir = '../checkpoints'
history = train_model(
    model,
    train_loader,
    val_loader,
    num_epochs=config['num_epochs'],
    lr=config['learning_rate'],
    device=device,
    checkpoint_dir=checkpoint_dir
)

print(f"\nTraining completed!")
print(f"Best validation accuracy: {max(history['val_acc']):.4f}")

## Plot Training History

In [ ]:
plot_history_path = '../data/training_history.png'
plot_training_history(history, save_path=plot_history_path)

print(f"\nFinal metrics:")
print(f"  Train Loss: {history['train_loss'][-1]:.4f}")
print(f"  Train Acc: {history['train_acc'][-1]:.4f}")
print(f"  Val Loss: {history['val_loss'][-1]:.4f}")
print(f"  Val Acc: {history['val_acc'][-1]:.4f}")

## Test Set Evaluation

In [ ]:
from src.evaluation.metrics import compute_classification_metrics

# Load best model
best_model_path = list(Path(checkpoint_dir).glob('best_model_acc_*.pth'))
if best_model_path:
    best_model_path = best_model_path[0]
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print(f"Loaded best model: {best_model_path}")

# Evaluate on test set
model.eval()
test_dataset = torch.utils.data.TensorDataset(
    torch.FloatTensor(test_specs_norm),
    torch.LongTensor(test_labels)
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

all_preds = []
all_labels = []

with torch.no_grad():
    for specs, labels in test_loader:
        specs = specs.to(device)
        outputs = model(specs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Compute metrics
metrics = compute_classification_metrics(all_labels, all_preds)
print(f"\nTest Set Performance:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

# Per-class accuracy
print(f"\nPer-class accuracy:")
for mod_idx, mod_name in label_to_mod.items():
    mask = all_labels == mod_idx
    if mask.sum() > 0:
        acc = (all_preds[mask] == all_labels[mask]).sum() / mask.sum()
        print(f"  {mod_name}: {acc:.4f}")

## Save Training Configuration

In [ ]:
config_path = '../data/training_config.json'
save_training_config(config, config_path)
print(f"Training configuration saved!")